# 📝 Notebook 03 — Claim Extraction
## VERA: Visual Evidence–Report Alignment

This notebook extracts structured anatomical claims from generated radiology reports.

**Steps:**
1. Load generated reports from Notebook 02
2. Run scispaCy NER + rule-based extraction
3. Extract: finding, location, severity, negation, token spans
4. Detect relational hallucinations
5. Save structured claims per image

## 1. Setup

In [ ]:
# Install NLP dependencies (uncomment for Colab/Kaggle)
# !pip install -q spacy scispacy
# !pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

import sys
import os
from pathlib import Path

# Add project root to path (auto-detect Kaggle vs local)
if os.path.exists('/kaggle/working'):
    PROJECT_ROOT = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from config import ATTENTION_DIR, CLAIMS_DIR, PROCESSED_DIR, FIGURES_DIR, IS_KAGGLE
from src.data_utils import load_json, save_json
from src.claim_extractor import (
    load_nlp_model, extract_claims, detect_relational_hallucinations,
    summarize_claims
)

import json
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from collections import Counter

## 2. Load NLP Model

In [ ]:
print("Loading NLP model...")
nlp_model = load_nlp_model("en_core_sci_sm")

if nlp_model is None:
    print("⚠️ scispaCy model not available. Using rule-based extraction only.")
    print("   This still works well for radiology reports!")
else:
    print(f"✅ NLP model loaded: {nlp_model.meta['name']}")

## 3. Load Generated Reports

In [ ]:
# Select model (must match Notebook 02)
USE_MODEL = "chexagent"  # or "llava_med"

# Load inference results
results_path = ATTENTION_DIR / USE_MODEL / 'inference_results.json'
if results_path.exists():
    inference_results = load_json(str(results_path))
    # Filter out failed entries
    inference_results = [r for r in inference_results if 'error' not in r]
    print(f"Loaded {len(inference_results)} inference results")
else:
    print(f"❌ No inference results found at {results_path}")
    print("   Run Notebook 02 first!")
    inference_results = []

# Show sample
if inference_results:
    sample = inference_results[0]
    print(f"\nSample generated report:")
    print(f"Image: {sample['image_id']}")
    print(f"Report: {sample.get('generated_report', '')[:300]}")

## 4. Extract Claims — Single Report Test

In [ ]:
# Test on a single report
if inference_results:
    test_report = inference_results[0].get('generated_report', '')
    print(f"Test report:\n{test_report}\n")
    print("="*60)
    
    # Extract claims
    test_claims = extract_claims(test_report, nlp_model, include_negated=True)
    
    print(f"\nExtracted {len(test_claims)} claims:")
    for i, claim in enumerate(test_claims):
        neg = " [NEGATED]" if claim['negated'] else ""
        print(f"  {i+1}. Finding: '{claim['finding']}' | "
              f"Location: '{claim['location']}' | "
              f"Severity: {claim['severity']}{neg}")
    
    # Check for relational hallucinations
    rel_flags = detect_relational_hallucinations(test_report)
    if rel_flags:
        print(f"\n⚠️ Relational hallucination flags:")
        for flag in rel_flags:
            print(f"   '{flag['matched_pattern']}' in: ...{flag['context']}...")
    else:
        print(f"\n✅ No relational hallucinations detected")

## 5. Batch Claim Extraction

In [ ]:
print("="*60)
print(f"Extracting claims from {len(inference_results)} reports")
print("="*60)

# Create output directory
claims_output_dir = CLAIMS_DIR / USE_MODEL
claims_output_dir.mkdir(parents=True, exist_ok=True)

all_claims_summary = []
total_claims = 0
total_relational = 0

for entry in tqdm(inference_results, desc="Extracting claims"):
    image_id = entry['image_id']
    report = entry.get('generated_report', '')
    
    if not report:
        continue
    
    # Extract claims
    claims = extract_claims(report, nlp_model, include_negated=True)
    
    # Detect relational hallucinations
    relational_flags = detect_relational_hallucinations(report)
    
    # Save per-image claims
    claims_data = {
        'image_id': image_id,
        'generated_report': report,
        'claims': claims,
        'relational_flags': relational_flags,
        'num_claims': len(claims),
        'num_relational_flags': len(relational_flags),
    }
    
    save_path = claims_output_dir / f"{image_id}_claims.json"
    with open(save_path, 'w') as f:
        json.dump(claims_data, f, indent=2)
    
    total_claims += len(claims)
    total_relational += len(relational_flags)
    all_claims_summary.append(claims_data)

print(f"\n✅ Extraction complete!")
print(f"   Total claims extracted: {total_claims}")
print(f"   Avg claims per report: {total_claims / len(inference_results):.1f}")
print(f"   Relational flags: {total_relational}")

## 6. Statistics & Visualization

In [ ]:
# Aggregate all claims for statistics
all_claims = []
for entry in all_claims_summary:
    all_claims.extend(entry['claims'])

summary = summarize_claims(all_claims)

print("\n" + "="*60)
print("CLAIM EXTRACTION STATISTICS")
print("="*60)
for key, value in summary.items():
    if isinstance(value, dict):
        print(f"\n  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Plot: Top findings distribution
findings_counter = Counter(c['finding'] for c in all_claims)
top_findings = findings_counter.most_common(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top findings
names, counts = zip(*top_findings) if top_findings else ([], [])
axes[0].barh(range(len(names)), counts, color='#3498db')
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(names)
axes[0].set_xlabel('Count')
axes[0].set_title('Top 15 Findings', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# Location distribution
locations_counter = Counter(c['location'] for c in all_claims if c['location'])
top_locations = locations_counter.most_common(10)
if top_locations:
    loc_names, loc_counts = zip(*top_locations)
    axes[1].barh(range(len(loc_names)), loc_counts, color='#2ecc71')
    axes[1].set_yticks(range(len(loc_names)))
    axes[1].set_yticklabels(loc_names)
    axes[1].set_xlabel('Count')
axes[1].set_title('Top 10 Anatomical Locations', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()

plt.suptitle('Claim Extraction Summary', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'claim_extraction_summary.png'), dpi=150)
plt.show()

In [ ]:
# Save overall summary
summary['model'] = USE_MODEL
summary['num_reports_processed'] = len(inference_results)
save_json(summary, str(claims_output_dir / 'claims_summary.json'))

print(f"\n📁 Claims saved to: {claims_output_dir}")
print(f"\nNext: Run 04_anatomy_atlas.ipynb (can run independently)")
print(f"Then: Run 05_vera_scoring.ipynb")